In [0]:
%run ./logger

### Step 1: Import Libraries

### Step 2: Create Database

### Step 3: Create Execution Log Table

database,tableName,isTemporary
medallion_db,execution_logs,false


### Step 4: Start Logger Function

### Step 5: Success Logger Function

### Step 6: Failure Logger Function

### Step 7: View Execution Logs

Batch_ID,Notebook_Name,Source_Layer,Target_Layer,Start_Time,End_Time,Duration_Seconds,Status,Records_Read,Records_Written,Error_Message
Batch_validation_001,validation,bronze,silver,2026-06-27T14:32:33.477234Z,2026-06-27T14:32:39.028798Z,5,SUCCESS,1017209,1017209,
Batch_raw_001,audit,raw,bronze,2026-06-27T14:27:16.548507Z,2026-06-27T14:27:19.125959Z,2,SUCCESS,1017209,1017209,
12345,gold_data,silver,gold,2026-06-27T14:26:16.802524Z,2026-06-27T14:26:59.794789Z,42,SUCCESS,1017209,1115,
12345,silver_data,bronze,silver,2026-06-27T14:25:39.522719Z,2026-06-27T14:26:00.783121Z,21,SUCCESS,1017209,1017209,
12345,bronze_data,raw,bronze,2026-06-27T14:24:55.630196Z,2026-06-27T14:25:18.673779Z,23,SUCCESS,1018324,1017209,
12345,raw_data,raw,bronze,2026-06-27T14:24:15.14494Z,2026-06-27T14:24:38.775364Z,23,SUCCESS,1018324,1018324,
Batch_slver_001,silver_data,bronze,silver,2026-06-27T14:22:56.658032Z,2026-06-27T14:23:16.399497Z,19,SUCCESS,1017209,1017209,
12345,bronze_data,raw,bronze,2026-06-27T14:04:23.259962Z,2026-06-27T14:04:46.96848Z,23,SUCCESS,1018324,1017209,
12345,raw_data,raw,bronze,2026-06-27T14:03:45.092331Z,2026-06-27T14:04:09.163528Z,24,SUCCESS,1018324,1018324,


### 

### 

In [0]:
from pyspark.sql.functions import *
from datetime import datetime
from pyspark.sql.functions import col

In [0]:
# start_time = log_start(batch_id, notebook_name, source_layer, target_layer)

In [0]:
batch_id = "Batch_validation_001"
try:
    batch_id = dbutils.widgets.get("batch_id")
except Exception:
    pass

if not batch_id:
    raise Exception("batch_id not passed from job")

In [0]:


notebook_name = "validation"
source_layer = "bronze"
target_layer = "silver"

start_time = log_start(batch_id, notebook_name, source_layer, target_layer)

try:

    df = spark.read.format("delta").load(
        "wasbs://bronze@insurancedatahomecredit.blob.core.windows.net/train"
    )

    total_count = df.count()
    unique_count = df.dropDuplicates().count()
    duplicate_count = total_count - unique_count

    if total_count == 0:
        raise Exception("Empty Data")

    log_success(
        batch_id,
        notebook_name,
        source_layer,
        target_layer,
        start_time,
        total_count,
        unique_count
    )

except Exception as e:

    log_failure(
        batch_id,
        notebook_name,
        source_layer,
        target_layer,
        start_time,
        0,
        0,
        str(e)
    )

    raise e